# Reactivity Pilot — Logit Measurement

범위: `reactivity_pilot/sync_reactivity_pilot_resynthesis_v0_2.md`에 정의된 reactivity pilot의
**logit 측정 단계만** 수행한다. activation patching은 이 노트북에 포함하지 않는다.

이 pilot은 `baseline_diagnostic/`의 baseline diagnostic과는 다른 실험이다 — 다른 데이터셋
(`dataset/reactivity_dataset.json`), 다른 metric(`LD_reactive`), clean/corrupted 쌍 없음. 두
실험의 결과를 섞어서 해석하지 않는다.

## 데이터셋 구조 (baseline과 다름)

- long format. **한 row = 한 cell = 하나의 prefix.**
- pair당 8 cell (2×2×2: `state_role` × `decl_order` × `body_order`).
- 한 prefix 안에 `reactive_var`, `stable_var` 두 변수가 모두 등장한다. clean/corrupted 쌍이
  아니라 **같은 prefix 안에서 두 변수의 logit을 비교**한다. corrupted prefix는 존재하지 않는다.
- 모든 prefix는 `[`로 끝나는 prefix-only다. target token은 입력에 없다. 측정 위치는 prefix의
  마지막 토큰(`[`) 위치다.

## 용어·주장 톤에 대한 중요한 주의

`reactive_var`/`stable_var`라는 이름은 **useState 선언 형태와 component-internal const 선언
형태에 붙인 라벨일 뿐**이며, 그 변수가 의미론적으로 reactive함을 측정했다는 뜻이 아니다. 이
노트북이 실제로 조작하는 것은 선언 형태(`const [x, setx] = useState(...)` vs `const x = ...`)이고,
측정하는 것은 그 선언 형태에 대한 모델의 선택 민감성이다.

다음과 같은 표현은 코드 주석, 로그 출력, markdown, 산출 문서 어디에도 쓰지 않는다:

- "LLM이 React reactivity를 이해한다"
- "React 의미론 회로"
- "useEffect 의미 이해"

이 노트북과 산출물에서 허용되는 가장 강한 표현은 **"useState-form / declared-reactivity
sensitivity가 관찰(되지 않)는다"** 정도다. positive 결과가 나와도 setter(`setX`)·destructuring
구조가 useState 형태에 항상 동반되므로, reactivity 자체와 그 부속 구조를 분리하지 못한다는 한계가
남는다 (상세는 `reactivity_summary.md`의 limitation 절, Step 10에서 작성).

## 실행 환경

- 필요 패키지: `transformer_lens`(`model_bridge.TransformerBridge`), `pandas`, `torch`.
- 위에서부터 순서대로 셀을 실행한다. 각 단계는 직전 단계의 출력(특히 assert 통과 여부)을 보고
  다음으로 넘어가도록 구성되어 있다.
- 이 노트북은 작성 시점에 GPU·모델 가중치가 없는 환경에서 작성되어 **실행 검증되지 않았다.**
  토큰화 가정, 모델별 tokenizer 차이, 측정 위치 등 깨지기 쉬운 지점에는 방어적 assert와 에러
  메시지를 넣었지만, 실제 실행 결과는 서버에서 셀을 돌리며 직접 확인해야 한다.


In [1]:
import json
from collections import Counter
from pathlib import Path

import pandas as pd
import torch

torch.set_grad_enabled(False)

from transformer_lens.model_bridge import TransformerBridge


## Configuration

모델명·경로는 모두 여기서만 바꾼다. `MODELS`의 두 항목을 그대로 두면 두 모델을 순회하는 구성으로
실행되며, 아래 셀들은 모델별로 쌍을 이루어 (Llama 먼저, Gemma 다음) 구성되어 있다.

이 셀을 실행하면 `dataset_full`이 만들어진다. **확인할 것:**

- 행 수가 360 (45 pair × 8 cell)인지
- 모든 pair가 정확히 8개 cell을 갖는지
- `DEVICE`가 의도한 값(서버라면 보통 `cuda`)으로 잡혔는지


In [2]:
DATASET_PATH = Path("../dataset/reactivity_dataset.json")

MODELS = {
    "llama3.2-1b": "meta-llama/Llama-3.2-1B",
    "gemma3-1b-pt": "google/gemma-3-1b-pt",
}
MODEL_LABELS = {"llama3.2-1b": "Llama 3.2 1B", "gemma3-1b-pt": "Gemma 3 1B-pt"}
MODEL_KEYS = list(MODELS.keys())

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

with open(DATASET_PATH) as f:
    dataset_full = json.load(f)

assert len(dataset_full) == 360, f"expected 360 rows (45 pair x 8 cell), got {len(dataset_full)}"

pair_cell_counts = Counter(row["pair_id"] for row in dataset_full)
bad_pairs = {pid: n for pid, n in pair_cell_counts.items() if n != 8}
assert not bad_pairs, f"pairs without exactly 8 cells: {bad_pairs}"

print(f"device = {DEVICE}")
print(f"dataset_full: {len(dataset_full)} rows, {len(pair_cell_counts)} pairs, all pairs have 8 cells")


device = cuda
dataset_full: 360 rows, 45 pairs, all pairs have 8 cells


## Helpers

target token id는 절대 하드코딩하지 않는다. 모든 id는 `model.to_tokens(prefix + suffix)`를 실제로
토큰화해서, prefix 대비 토큰 수가 정확히 늘어난 만큼만 늘었는지 `assert`로 확인한 뒤 마지막 토큰을
가져온다.

- **authoritative target은 row의 `reactive_var`/`stable_var` 필드(bare identifier)다.**
  `vars.dep`/`vars.alt`는 generator 내부 배정용 기록일 뿐이고, 이 둘 중 어느 것이
  `reactive_var`인지는 row마다 다르다. `vars.dep`/`vars.alt`를 직접 타겟으로 쓰지 않는다.
- **Stage 1 기준(필수, 실패 시 해당 pair 전체 제외):** `reactive_id`/`stable_id` 둘 다 prefix
  대비 정확히 +1 토큰이고(single-token), `reactive_id != stable_id`.
- `close_id`(`]`)는 **auxiliary 전용**이며 main 판정에서 절대 쓰지 않는다. baseline에서 이미
  확인된 바와 같이, prefix가 ` [`로 끝나기 때문에 `]`를 바로 이어붙이면 ` []`라는 단일 병합
  토큰이 되어 +1 토큰 assert가 깨진다(두 토크나이저 모두 `"[]"`가 단일 vocab 항목). 대신
  `prefix + reactive_var + "]"`로 닫힌 1-원소 배열 문맥을 만들어 +2 토큰으로 검증한다(sync note
  §10.2). 이 close 계산이 실패해도 해당 row를 Stage 1에서 탈락시키지 않는다 — `logit_close`만
  NaN으로 남기고 main 측정은 계속한다.


In [3]:
def get_target_id(model, prefix, prefix_n, suffix, expected_add):
    """Token id for `prefix + suffix`, asserted to add exactly `expected_add` tokens."""
    full_tokens = model.to_tokens(prefix + suffix)
    n = full_tokens.shape[-1]
    assert n == prefix_n + expected_add, (
        f"token count mismatch: prefix_n={prefix_n} suffix={suffix!r} "
        f"got n={n} expected={prefix_n + expected_add}"
    )
    return full_tokens[0, -1].item()


def validate_and_get_ids(model, prefix, reactive_var, stable_var):
    """
    Stage-1 check for one row. Returns (prefix_n, reactive_id, stable_id).
    Raises AssertionError if: last prefix token isn't '[', either id isn't a
    single-token continuation of prefix, or reactive_id == stable_id.
    """
    prefix_tokens = model.to_tokens(prefix)
    prefix_n = prefix_tokens.shape[-1]

    str_tokens = model.to_str_tokens(prefix)
    assert str_tokens[-1].strip() == "[", (
        f"last prefix token isn't '[': {str_tokens[-1]!r}"
    )

    reactive_id = get_target_id(model, prefix, prefix_n, reactive_var, 1)
    stable_id = get_target_id(model, prefix, prefix_n, stable_var, 1)
    assert reactive_id != stable_id, (
        f"reactive_id == stable_id ({reactive_id}) for "
        f"reactive_var={reactive_var!r} stable_var={stable_var!r}"
    )
    return prefix_n, reactive_id, stable_id


def get_close_id(model, prefix, prefix_n, reactive_var):
    """
    Auxiliary only (never the main metric). See markdown above for why
    `prefix + reactive_var + "]"` is used instead of `prefix + "]"`.
    """
    return get_target_id(model, prefix, prefix_n, reactive_var + "]", 2)


## Step 1 — Llama 3.2 1B 로드 및 1-row 검증

모델을 로드하고, dataset의 첫 row 하나에 대해서만 토큰화 assert와 forward pass가 모두 문제없이
도는지 확인한다. 여기서 에러가 나면 전체 데이터셋을 돌리기 전에 토큰화 가정이나 모델 로딩이
깨졌다는 뜻이므로 먼저 원인을 봐야 한다.

(`meta-llama/Llama-3.2-1B`는 gated repo다. 로딩이 권한 에러로 실패하면 HF 토큰/로그인 설정을
먼저 확인하라.)


In [4]:
print(f"Loading model: {MODELS['llama3.2-1b']}")
model_llama = TransformerBridge.boot_transformers(MODELS["llama3.2-1b"], device=DEVICE)
print("Loaded.")


Loading model: meta-llama/Llama-3.2-1B


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loaded.


In [5]:
row0 = dataset_full[0]

prefix_n0, reactive_id0, stable_id0 = validate_and_get_ids(
    model_llama, row0["prefix"], row0["reactive_var"], row0["stable_var"]
)
close_id0 = get_close_id(model_llama, row0["prefix"], prefix_n0, row0["reactive_var"])

# Minimal forward-pass check here (not just tokenization) so that a
# device/shape problem surfaces on 1 row, not 360 rows later.
prefix_tokens0 = model_llama.to_tokens(row0["prefix"])
logits0 = model_llama(prefix_tokens0)
assert logits0.shape[0] == 1 and logits0.shape[1] == prefix_n0, (
    f"unexpected logits shape {tuple(logits0.shape)} for prefix_n={prefix_n0}"
)

print(f"row={row0['id']} reactive_var={row0['reactive_var']!r} stable_var={row0['stable_var']!r}")
print(f"prefix_n={prefix_n0} reactive_id={reactive_id0} stable_id={stable_id0} close_id={close_id0}")
print(f"logits shape={tuple(logits0.shape)}")
print("ASSERTIONS PASSED (Llama 3.2 1B, 1 row, incl. forward pass)")


row=pair_01_c1 reactive_var='page' stable_var='ref'
prefix_n=45 reactive_id=2964 stable_id=1116 close_id=60
logits shape=(1, 45, 128256)
ASSERTIONS PASSED (Llama 3.2 1B, 1 row, incl. forward pass)


## Step 2 — Gemma 3 1B-pt 로드 및 1-row 검증

같은 row(`row0`)로 Gemma 쪽도 동일하게 확인한다. 두 모델의 tokenizer가 다르므로 토큰 id 값은
달라도 되지만, assert와 forward pass는 둘 다 통과해야 한다.


In [6]:
print(f"Loading model: {MODELS['gemma3-1b-pt']}")
model_gemma = TransformerBridge.boot_transformers(MODELS["gemma3-1b-pt"], device=DEVICE)
print("Loaded.")


Loading model: google/gemma-3-1b-pt


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Loaded.


In [7]:
prefix_n0_g, reactive_id0_g, stable_id0_g = validate_and_get_ids(
    model_gemma, row0["prefix"], row0["reactive_var"], row0["stable_var"]
)
close_id0_g = get_close_id(model_gemma, row0["prefix"], prefix_n0_g, row0["reactive_var"])

prefix_tokens0_g = model_gemma.to_tokens(row0["prefix"])
logits0_g = model_gemma(prefix_tokens0_g)
assert logits0_g.shape[0] == 1 and logits0_g.shape[1] == prefix_n0_g, (
    f"unexpected logits shape {tuple(logits0_g.shape)} for prefix_n={prefix_n0_g}"
)

print(f"row={row0['id']} reactive_var={row0['reactive_var']!r} stable_var={row0['stable_var']!r}")
print(f"prefix_n={prefix_n0_g} reactive_id={reactive_id0_g} stable_id={stable_id0_g} close_id={close_id0_g}")
print(f"logits shape={tuple(logits0_g.shape)}")
print("ASSERTIONS PASSED (Gemma 3 1B-pt, 1 row, incl. forward pass)")


row=pair_01_c1 reactive_var='page' stable_var='ref'
prefix_n=51 reactive_id=4000 stable_id=1811 close_id=236842
logits shape=(1, 51, 262144)
ASSERTIONS PASSED (Gemma 3 1B-pt, 1 row, incl. forward pass)


## Step 3 — Stage 1: 전체 dataset 토큰 검증 및 pair 제외

360 row 전체에 대해 두 모델 각각 `validate_and_get_ids`를 시도한다. **두 모델 중 하나라도 row
하나에서 실패하면, 그 row가 속한 pair의 8 cell 전체를 이후 모든 분석에서 제외한다**(sync note
지시사항 — pair 단위 제외, row 단위 제외가 아니다). 제외된 pair는 사유와 함께 출력하고, 이후
셀에서 쓰는 `dataset_valid`는 이 필터링을 거친 데이터셋이다.

**확인할 것:** 통과율, 제외된 pair가 있다면 그 사유(어느 모델에서, 어떤 assert가 실패했는지).


In [8]:
def stage1_validate(model, rows):
    failures = []
    for row in rows:
        try:
            validate_and_get_ids(model, row["prefix"], row["reactive_var"], row["stable_var"])
        except AssertionError as e:
            failures.append({"id": row["id"], "pair_id": row["pair_id"], "error": str(e)})
    return len(rows) - len(failures), len(rows), failures


In [9]:
n_ok_llama, n_total, failures_llama = stage1_validate(model_llama, dataset_full)
print(f"Llama 3.2 1B: {n_ok_llama}/{n_total} rows passed")
for f in failures_llama:
    print("  FAIL", f)


Llama 3.2 1B: 360/360 rows passed


In [10]:
n_ok_gemma, n_total, failures_gemma = stage1_validate(model_gemma, dataset_full)
print(f"Gemma 3 1B-pt: {n_ok_gemma}/{n_total} rows passed")
for f in failures_gemma:
    print("  FAIL", f)


Gemma 3 1B-pt: 360/360 rows passed


In [11]:
failed_pair_ids = sorted(
    {f["pair_id"] for f in failures_llama} | {f["pair_id"] for f in failures_gemma}
)

dataset_valid = [row for row in dataset_full if row["pair_id"] not in failed_pair_ids]

n_pairs_total = len({row["pair_id"] for row in dataset_full})
n_pairs_excluded = len(failed_pair_ids)
n_pairs_kept = n_pairs_total - n_pairs_excluded

print(f"pairs total={n_pairs_total} excluded={n_pairs_excluded} kept={n_pairs_kept}")
print(f"dataset_valid: {len(dataset_valid)} rows ({n_pairs_kept} pair x 8 cell)")
if failed_pair_ids:
    print("excluded pair_ids:", failed_pair_ids)
else:
    print("no pair excluded -- token validation passed for all rows in both models")

assert len(dataset_valid) == n_pairs_kept * 8, "dataset_valid row count doesn't match pair count x 8"


pairs total=45 excluded=0 kept=45
dataset_valid: 360 rows (45 pair x 8 cell)
no pair excluded -- token validation passed for all rows in both models


## Step 4 — 소수 cell dry run (첫 valid pair의 8 cell)

전체 측정 전에, `dataset_valid`의 첫 pair(8 cell)에 대해서만 측정을 돌려 눈으로 확인한다.

**확인할 것:**

- `logit_reactive`, `logit_stable`이 NaN이 아닌 유한값인지
- 같은 pair 안에서 8 cell의 `state_role`/`decl_order`/`body_order`/`reactive_body_pos` 조합이
  8가지 모두 다른지 (중복 없이 2×2×2)
- `close_error`가 채워진 row가 있는지 (있어도 main 측정에는 영향 없음 — `logit_close`만 NaN)
- 이 시점에서는 `LD_reactive`의 부호를 어떤 방향으로든 미리 판단하지 않는다. dry run의 목적은
  파이프라인이 깨지지 않고 도는지 확인하는 것이고, 패턴 판단은 전체 측정 후 Step 7–9에서 한다.


In [12]:
def measure_rows(model, model_key, rows):
    out = []
    for row in rows:
        prefix = row["prefix"]
        reactive_var, stable_var = row["reactive_var"], row["stable_var"]

        prefix_n, reactive_id, stable_id = validate_and_get_ids(model, prefix, reactive_var, stable_var)

        close_id, close_error = None, None
        try:
            close_id = get_close_id(model, prefix, prefix_n, reactive_var)
        except AssertionError as e:
            close_error = str(e)

        prefix_tokens = model.to_tokens(prefix)
        logits = model(prefix_tokens)
        pos = prefix_n - 1
        pos_logits = logits[0, pos, :]

        logit_reactive = pos_logits[reactive_id].item()
        logit_stable = pos_logits[stable_id].item()
        logit_close = pos_logits[close_id].item() if close_id is not None else float("nan")

        out.append({
            "model": model_key,
            "pair_id": row["pair_id"],
            "cell_id": row["id"],
            "state_role": row["state_role"],
            "decl_order": row["decl_order"],
            "body_order": row["body_order"],
            "reactive_var": reactive_var,
            "stable_var": stable_var,
            "reactive_body_pos": row["reactive_body_pos"],
            "prefix": prefix,
            "n_tokens": prefix_n,
            "reactive_id": reactive_id,
            "stable_id": stable_id,
            "close_id": close_id,
            "logit_reactive": logit_reactive,
            "logit_stable": logit_stable,
            "logit_close": logit_close,
            "LD_reactive": logit_reactive - logit_stable,
            "LD_reactive_vs_close": logit_reactive - logit_close,
            "LD_stable_vs_close": logit_stable - logit_close,
            "close_error": close_error,
        })
    return out


In [13]:
first_pair_id = dataset_valid[0]["pair_id"]
dry_rows = [row for row in dataset_valid if row["pair_id"] == first_pair_id]
assert len(dry_rows) == 8, f"expected 8 cells for {first_pair_id}, got {len(dry_rows)}"

dry_df_llama = pd.DataFrame(measure_rows(model_llama, "llama3.2-1b", dry_rows))
dry_df_llama[[
    "cell_id", "state_role", "decl_order", "body_order", "reactive_body_pos",
    "reactive_var", "stable_var", "logit_reactive", "logit_stable", "logit_close",
    "LD_reactive", "close_error",
]]


,cell_id,state_role,decl_order,body_order,reactive_body_pos,reactive_var,stable_var,logit_reactive,logit_stable,logit_close,LD_reactive,close_error
0,pair_01_c1,dep_reactive,reactive_first,dep_first,first,page,ref,26.247749,24.119148,12.353183,2.128601,None
1,pair_01_c2,dep_reactive,reactive_first,alt_first,second,page,ref,25.896322,25.812880,12.245905,0.083443,None
2,pair_01_c3,dep_reactive,stable_first,dep_first,first,page,ref,26.298126,24.719400,12.167810,1.578726,None
3,pair_01_c4,dep_reactive,stable_first,alt_first,second,page,ref,25.834534,26.051178,12.116523,-0.216644,None
4,pair_01_c5,alt_reactive,reactive_first,dep_first,second,ref,page,25.989292,26.502106,12.580136,-0.512814,None
5,pair_01_c6,alt_reactive,reactive_first,alt_first,first,ref,page,27.131882,25.361649,12.531300,1.770233,None
6,pair_01_c7,alt_reactive,stable_first,dep_first,second,ref,page,25.971001,26.791321,12.832000,-0.820320,None
7,pair_01_c8,alt_reactive,stable_first,alt_first,first,ref,page,27.312035,25.802732,12.857705,1.509302,None


In [14]:
dry_df_gemma = pd.DataFrame(measure_rows(model_gemma, "gemma3-1b-pt", dry_rows))
dry_df_gemma[[
    "cell_id", "state_role", "decl_order", "body_order", "reactive_body_pos",
    "reactive_var", "stable_var", "logit_reactive", "logit_stable", "logit_close",
    "LD_reactive", "close_error",
]]


,cell_id,state_role,decl_order,body_order,reactive_body_pos,reactive_var,stable_var,logit_reactive,logit_stable,logit_close,LD_reactive,close_error
0,pair_01_c1,dep_reactive,reactive_first,dep_first,first,page,ref,21.901867,19.639921,8.268684,2.261946,None
1,pair_01_c2,dep_reactive,reactive_first,alt_first,second,page,ref,21.083719,20.865421,7.976949,0.218298,None
2,pair_01_c3,dep_reactive,stable_first,dep_first,first,page,ref,22.036140,20.375572,8.072012,1.660568,None
3,pair_01_c4,dep_reactive,stable_first,alt_first,second,page,ref,20.712551,21.165987,7.885341,-0.453436,None
4,pair_01_c5,alt_reactive,reactive_first,dep_first,second,ref,page,20.721111,21.532066,8.360812,-0.810955,None
5,pair_01_c6,alt_reactive,reactive_first,alt_first,first,ref,page,21.250187,19.914057,8.162783,1.336130,None
6,pair_01_c7,alt_reactive,stable_first,dep_first,second,ref,page,20.893265,21.821999,8.641184,-0.928734,None
7,pair_01_c8,alt_reactive,stable_first,alt_first,first,ref,page,21.677877,20.858549,8.390153,0.819328,None


## Step 5 — Stage 2: 전체 측정 (`dataset_valid` 전체 × 2 모델)

dry run이 정상이면 `dataset_valid` 전체에 대해 같은 `measure_rows`를 돌린다. row 수만큼
forward pass가 1회씩 도므로, 모델별 `len(dataset_valid)`번의 forward pass가 일어난다.


In [15]:
rows_llama = measure_rows(model_llama, "llama3.2-1b", dataset_valid)
print(f"llama3.2-1b: {len(rows_llama)} rows measured")


llama3.2-1b: 360 rows measured


In [16]:
rows_gemma = measure_rows(model_gemma, "gemma3-1b-pt", dataset_valid)
print(f"gemma3-1b-pt: {len(rows_gemma)} rows measured")


gemma3-1b-pt: 360 rows measured


In [17]:
df_long = pd.DataFrame(rows_llama + rows_gemma)
print(df_long.shape)
df_long.to_csv("reactivity_logits_3tokens.csv", index=False)
df_long.head()


(720, 21)


,model,pair_id,cell_id,state_role,decl_order,body_order,reactive_var,stable_var,reactive_body_pos,prefix,...,reactive_id,stable_id,close_id,logit_reactive,logit_stable,logit_close,LD_reactive,LD_reactive_vs_close,LD_stable_vs_close,close_error
0,llama3.2-1b,pair_01,pair_01_c1,dep_reactive,reactive_first,dep_first,page,ref,first,"function Component() {\n const [page, setPage...",...,2964,1116,60,26.247749,24.119148,12.353183,2.128601,13.894567,11.765965,None
1,llama3.2-1b,pair_01,pair_01_c2,dep_reactive,reactive_first,alt_first,page,ref,second,"function Component() {\n const [page, setPage...",...,2964,1116,60,25.896322,25.812880,12.245905,0.083443,13.650417,13.566975,None
2,llama3.2-1b,pair_01,pair_01_c3,dep_reactive,stable_first,dep_first,page,ref,first,function Component() {\n const ref = 1;\n co...,...,2964,1116,60,26.298126,24.719400,12.167810,1.578726,14.130316,12.551590,None
3,llama3.2-1b,pair_01,pair_01_c4,dep_reactive,stable_first,alt_first,page,ref,second,function Component() {\n const ref = 1;\n co...,...,2964,1116,60,25.834534,26.051178,12.116523,-0.216644,13.718011,13.934655,None
4,llama3.2-1b,pair_01,pair_01_c5,alt_reactive,reactive_first,dep_first,ref,page,second,"function Component() {\n const [ref, setRef] ...",...,1116,2964,60,25.989292,26.502106,12.580136,-0.512814,13.409156,13.921969,None


## Step 6 — Stage 3: pair 단위 wide table + factor effect

`dataset_valid`의 각 pair는 8 cell을 갖는다. 같은 pair의 8 cell을 나란히 두고, 동시에 세 factor
(state_role / decl_order / body_order)와 reactive_body_pos의 main effect를 pair 단위로
계산한다. effect는 그 factor의 한쪽 레벨 평균 `LD_reactive`에서 반대쪽 레벨 평균을 뺀 값이다
(예: `effect_state_role` = mean(dep_reactive cells) − mean(alt_reactive cells), pair 내 4
cell씩 평균).


In [18]:
def combo_label(row):
    return f"sr={row['state_role']}|do={row['decl_order']}|bo={row['body_order']}"


df_long["combo"] = df_long.apply(combo_label, axis=1)

wide = df_long.pivot_table(
    index=["model", "pair_id"],
    columns="combo",
    values=["LD_reactive", "logit_reactive", "logit_stable", "logit_close"],
)
wide.columns = [f"{a}__{b}" for a, b in wide.columns]
wide = wide.reset_index()


def compute_factor_effects(df):
    records = []
    for (model_key, pair_id), g in df.groupby(["model", "pair_id"]):
        records.append({
            "model": model_key,
            "pair_id": pair_id,
            "n_cells": len(g),
            "mean_LD_reactive_pair": g["LD_reactive"].mean(),
            "effect_state_role": (
                g.loc[g["state_role"] == "dep_reactive", "LD_reactive"].mean()
                - g.loc[g["state_role"] == "alt_reactive", "LD_reactive"].mean()
            ),
            "effect_decl_order": (
                g.loc[g["decl_order"] == "reactive_first", "LD_reactive"].mean()
                - g.loc[g["decl_order"] == "stable_first", "LD_reactive"].mean()
            ),
            "effect_body_order": (
                g.loc[g["body_order"] == "dep_first", "LD_reactive"].mean()
                - g.loc[g["body_order"] == "alt_first", "LD_reactive"].mean()
            ),
            "effect_reactive_body_pos": (
                g.loc[g["reactive_body_pos"] == "first", "LD_reactive"].mean()
                - g.loc[g["reactive_body_pos"] == "second", "LD_reactive"].mean()
            ),
        })
    return pd.DataFrame(records)


factor_effects = compute_factor_effects(df_long)
assert (factor_effects["n_cells"] == 8).all(), "every kept pair must have exactly 8 cells per model"

wide_per_sample = wide.merge(factor_effects, on=["model", "pair_id"])
wide_per_sample.to_csv("reactivity_wide_per_sample.csv", index=False)
wide_per_sample.shape


(90, 40)

## Step 7 — Stage 3: 모델별 전체 / factor별 / 완전교차 집계

세 단계로 본다.

1. **전체(overall):** 모델별 전체 valid cell에 대한 `LD_reactive`의 median/mean/frac(>0).
2. **단일 factor 분해(marginal):** state_role / decl_order / body_order / reactive_body_pos
   각각 단독으로 나눈 marginal 집계. 다른 두 factor는 평균으로 깔려 있다.
3. **완전교차(fully-crossed):** `model × state_role × decl_order × body_order ×
   reactive_body_pos`로 나눈 표 — sync note §11의 `reactivity_summary_table.csv` 스펙 그대로다.
   (`reactive_body_pos`는 `state_role`과 `body_order`로 결정되는 라벨이라, 실제 고유 조합 수는
   모델당 16이 아니라 8개다.)

**`reactive_body_pos`별 분해가 가장 중요한 통제다** — `LD_reactive`가 양수여도 그것이 useState
형태 때문인지, 단순히 body에서 먼저/나중에 나온 변수를 선호하는 order bias 때문인지 여기서
갈린다.


In [19]:
def agg_block(df, group_cols):
    return (
        df.groupby(group_cols)
        .agg(
            n=("LD_reactive", "size"),
            median_LD_reactive=("LD_reactive", "median"),
            mean_LD_reactive=("LD_reactive", "mean"),
            frac_LD_reactive_pos=("LD_reactive", lambda s: (s > 0).mean()),
        )
        .reset_index()
    )


overall_summary = agg_block(df_long, ["model"])
by_state_role = agg_block(df_long, ["model", "state_role"])
by_decl_order = agg_block(df_long, ["model", "decl_order"])
by_body_order = agg_block(df_long, ["model", "body_order"])
by_reactive_body_pos = agg_block(df_long, ["model", "reactive_body_pos"])

GROUP_COLS = ["model", "state_role", "decl_order", "body_order", "reactive_body_pos"]
summary_table = (
    df_long.groupby(GROUP_COLS)
    .agg(
        n=("LD_reactive", "size"),
        median_LD_reactive=("LD_reactive", "median"),
        mean_LD_reactive=("LD_reactive", "mean"),
        frac_LD_reactive_pos=("LD_reactive", lambda s: (s > 0).mean()),
        median_LD_reactive_vs_close=("LD_reactive_vs_close", "median"),
        median_LD_stable_vs_close=("LD_stable_vs_close", "median"),
    )
    .reset_index()
)
summary_table.to_csv("reactivity_summary_table.csv", index=False)

print("=== overall ===")
print(overall_summary)
print("\n=== by state_role ===")
print(by_state_role)
print("\n=== by decl_order ===")
print(by_decl_order)
print("\n=== by body_order ===")
print(by_body_order)
print("\n=== by reactive_body_pos ===")
print(by_reactive_body_pos)
print(f"\nfully-crossed summary_table: {summary_table.shape[0]} rows (expect up to {2 * 8} = 16, 8 distinct combos x 2 models)")
summary_table


=== overall ===
          model    n  median_LD_reactive  mean_LD_reactive  \
0  gemma3-1b-pt  360            0.437702          0.360482   
1   llama3.2-1b  360            1.092321          1.071337   

   frac_LD_reactive_pos  
0              0.602778  
1              0.727778  

=== by state_role ===
          model    state_role    n  median_LD_reactive  mean_LD_reactive  \
0  gemma3-1b-pt  alt_reactive  180           -0.117832         -0.228935   
1  gemma3-1b-pt  dep_reactive  180            0.908355          0.949899   
2   llama3.2-1b  alt_reactive  180            0.576706          0.631643   
3   llama3.2-1b  dep_reactive  180            1.524052          1.511031   

   frac_LD_reactive_pos  
0              0.472222  
1              0.733333  
2              0.611111  
3              0.844444  

=== by decl_order ===
          model      decl_order    n  median_LD_reactive  mean_LD_reactive  \
0  gemma3-1b-pt  reactive_first  180            0.648606          0.560695   
1  gem

,model,state_role,decl_order,body_order,reactive_body_pos,n,median_LD_reactive,mean_LD_reactive,frac_LD_reactive_pos,median_LD_reactive_vs_close,median_LD_stable_vs_close
0,gemma3-1b-pt,alt_reactive,reactive_first,alt_first,first,45,1.395149,1.336528,0.933333,12.938468,11.611357
1,gemma3-1b-pt,alt_reactive,reactive_first,dep_first,second,45,-1.405172,-1.250728,0.066667,11.811404,13.171254
2,gemma3-1b-pt,alt_reactive,stable_first,alt_first,first,45,0.753550,0.643052,0.866667,12.916344,12.336703
3,gemma3-1b-pt,alt_reactive,stable_first,dep_first,second,45,-1.696676,-1.644592,0.022222,11.494069,13.180815
4,gemma3-1b-pt,dep_reactive,reactive_first,alt_first,second,45,0.114435,0.026785,0.555556,12.612957,12.800400
5,gemma3-1b-pt,dep_reactive,reactive_first,dep_first,first,45,2.159061,2.130194,1.000000,13.482415,11.371237
6,gemma3-1b-pt,dep_reactive,stable_first,alt_first,second,45,-0.165951,-0.262240,0.377778,12.426167,12.937873
7,gemma3-1b-pt,dep_reactive,stable_first,dep_first,first,45,1.802717,1.904855,1.000000,13.476479,11.853054
8,llama3.2-1b,alt_reactive,reactive_first,alt_first,first,45,2.457933,2.423729,1.000000,13.779913,11.330973
9,llama3.2-1b,alt_reactive,reactive_first,dep_first,second,45,-0.349480,-0.202538,0.377778,13.138900,13.103716


## Step 8 — Auxiliary: close(`]`) 진단

`logit_close`/`LD_reactive_vs_close`/`LD_stable_vs_close`는 **main 해석에 쓰지 않는다.** 여기서는
close 계산 실패율과, reactive/stable 변수가 실제로 `]`보다 더 선호되는 next-token candidate로
보이는지만 참고용으로 본다.


In [20]:
aux_rows = []
for model_key, g in df_long.groupby("model"):
    g_ok = g.dropna(subset=["logit_close"])
    aux_rows.append({
        "model": model_key,
        "n_total": len(g),
        "n_close_ok": len(g_ok),
        "n_close_fail": len(g) - len(g_ok),
        "median_LD_reactive_vs_close": g_ok["LD_reactive_vs_close"].median() if len(g_ok) else float("nan"),
        "median_LD_stable_vs_close": g_ok["LD_stable_vs_close"].median() if len(g_ok) else float("nan"),
        "frac_reactive_below_close": (g_ok["LD_reactive_vs_close"] < 0).mean() if len(g_ok) else float("nan"),
        "frac_stable_below_close": (g_ok["LD_stable_vs_close"] < 0).mean() if len(g_ok) else float("nan"),
    })

aux_close_summary = pd.DataFrame(aux_rows)
aux_close_summary


,model,n_total,n_close_ok,n_close_fail,median_LD_reactive_vs_close,median_LD_stable_vs_close,frac_reactive_below_close,frac_stable_below_close
0,gemma3-1b-pt,360,360,0,12.757554,12.355656,0.0,0.0
1,llama3.2-1b,360,360,0,13.468239,12.468183,0.0,0.0


## Step 9 — order-bias 대 role-effect 자동 비교 (초안, 최종 해석 아님)

`reactive_body_pos`(또는 `body_order`)의 marginal effect 크기가 `state_role`의 marginal effect
크기보다 크면, order-dominated negative result 쪽 해석에 가깝다(sync note §9.2). 아래는 이
비교를 자동으로 계산한 초안이며, **최종 해석과 메인 claim 선택은 연구자가 한다.**


In [21]:
def marginal_range(agg_df):
    return agg_df["median_LD_reactive"].max() - agg_df["median_LD_reactive"].min()


flag_rows = []
for model_key in MODEL_KEYS:
    range_state_role = marginal_range(by_state_role[by_state_role["model"] == model_key])
    range_decl_order = marginal_range(by_decl_order[by_decl_order["model"] == model_key])
    range_body_order = marginal_range(by_body_order[by_body_order["model"] == model_key])
    range_reactive_body_pos = marginal_range(by_reactive_body_pos[by_reactive_body_pos["model"] == model_key])

    model_summary = summary_table[summary_table["model"] == model_key]
    sign_consistent = bool(
        (model_summary["median_LD_reactive"] > 0).all()
        or (model_summary["median_LD_reactive"] < 0).all()
    )

    flag_rows.append({
        "model": model_key,
        "range_state_role": range_state_role,
        "range_decl_order": range_decl_order,
        "range_body_order": range_body_order,
        "range_reactive_body_pos": range_reactive_body_pos,
        "order_range_exceeds_role_range": bool(
            max(range_body_order, range_reactive_body_pos) > range_state_role
        ),
        "median_LD_reactive_sign_consistent_across_8_combos": sign_consistent,
        "overall_median_LD_reactive": overall_summary.loc[
            overall_summary["model"] == model_key, "median_LD_reactive"
        ].iloc[0],
        "overall_frac_LD_reactive_pos": overall_summary.loc[
            overall_summary["model"] == model_key, "frac_LD_reactive_pos"
        ].iloc[0],
    })

order_vs_role_flag = pd.DataFrame(flag_rows)
order_vs_role_flag


,model,range_state_role,range_decl_order,range_body_order,range_reactive_body_pos,order_range_exceeds_role_range,median_LD_reactive_sign_consistent_across_8_combos,overall_median_LD_reactive,overall_frac_LD_reactive_pos
0,llama3.2-1b,0.947346,0.624332,0.013486,2.464013,True,False,1.092321,0.727778
1,gemma3-1b-pt,1.026187,0.430205,0.140324,2.173289,True,False,0.437702,0.602778


## Step 10 — `reactivity_summary.md` 작성

지금까지 계산한 모든 표(token validation 결과, 전체/factor/완전교차 집계, order-vs-role
자동 비교, auxiliary close 진단)를 하나의 markdown 문서로 정리한다. 해석 문장은 sync note §9
가이드를 그대로 따르며 수치와 함께 제시하되, 최종 판단으로 단정하지 않는다.


In [22]:
def fmt(x):
    if pd.isna(x):
        return "NaN"
    return f"{x:.3f}"


lines = []
lines.append("# Reactivity Pilot -- Logit Measurement Summary")
lines.append("")
lines.append(
    "범위: reactivity pilot의 logit 측정 단계(`sync_reactivity_pilot_resynthesis_v0_2.md` "
    "§10-§11)만 다룬다. activation patching은 포함하지 않는다. baseline diagnostic "
    "(`baseline_diagnostic/`)과는 다른 데이터셋ㆍ다른 metric을 쓰는 별도 실험이다."
)
lines.append("")
lines.append(
    "`reactive_var`/`stable_var`는 useState/const 선언 형태에 붙인 라벨이며, reactivity 그 "
    "자체를 측정한 것이 아니다. 이 문서가 지지하는 가장 강한 주장은 \"useState-form / "
    "declared-reactivity sensitivity\"이며, \"React reactivity 의미 이해\" 수준의 주장은 "
    "하지 않는다."
)
lines.append("")

lines.append("## 0. 데이터셋 및 제외 내역 (Stage 1)")
lines.append("")
lines.append(
    f"원본 데이터셋: {n_pairs_total} pair x 8 cell = {len(dataset_full)} row. "
    f"두 모델(Llama 3.2 1B, Gemma 3 1B-pt) 모두에서 row 단위로 reactive_var/stable_var "
    f"single-token 여부와 reactive_id != stable_id를 확인했다."
)
lines.append("")
lines.append(f"- Llama 3.2 1B: {n_ok_llama}/{n_total} row 통과")
lines.append(f"- Gemma 3 1B-pt: {n_ok_gemma}/{n_total} row 통과")
lines.append("")
if failed_pair_ids:
    lines.append(
        f"두 모델 중 하나라도 실패한 row가 속한 pair {n_pairs_excluded}개를 전체 분석에서 "
        f"제외했다(해당 pair의 8 cell 전체): {', '.join(failed_pair_ids)}."
    )
    lines.append("")
    lines.append("| pair_id | 실패 모델 | 실패 row | 사유 |")
    lines.append("|---|---|---|---|")
    for f in failures_llama:
        lines.append(f"| {f['pair_id']} | Llama 3.2 1B | {f['id']} | {f['error']} |")
    for f in failures_gemma:
        lines.append(f"| {f['pair_id']} | Gemma 3 1B-pt | {f['id']} | {f['error']} |")
    lines.append("")
else:
    lines.append("두 모델 모두 전체 row에서 통과했고, 제외된 pair는 없다.")
    lines.append("")
lines.append(
    f"이후 분석은 모두 `dataset_valid` ({n_pairs_kept} pair x 8 cell = {len(dataset_valid)} row) "
    f"기준이다."
)
lines.append("")

lines.append("## 1. 전체 결과 (Stage 3 -- overall)")
lines.append("")
lines.append(
    "`LD_reactive = logit(reactive_var) - logit(stable_var)`. 양수면 같은 prefix 안에서 모델이 "
    "const 형태보다 useState 형태 식별자에 더 높은 logit을 두었다는 뜻이다."
)
lines.append("")
lines.append("| model | n | median LD_reactive | mean LD_reactive | frac(LD_reactive>0) |")
lines.append("|---|---|---|---|---|")
for _, r in overall_summary.iterrows():
    lines.append(
        f"| {MODEL_LABELS[r['model']]} | {int(r['n'])} | {fmt(r['median_LD_reactive'])} | "
        f"{fmt(r['mean_LD_reactive'])} | {r['frac_LD_reactive_pos']:.3f} |"
    )
lines.append("")

lines.append("## 2. Factor별 분해 (marginal)")
lines.append("")
lines.append(
    "각 factor를 단독으로 나눈 marginal 집계다(다른 두 factor는 평균으로 깔려 있음). 완전교차 "
    "표는 §4와 `reactivity_summary_table.csv`에 별도로 있다."
)
lines.append("")


def factor_table(title, agg_df, factor_col):
    out = [f"### {title}", ""]
    out.append(
        f"| model | {factor_col} | n | median LD_reactive | mean LD_reactive | "
        f"frac(LD_reactive>0) |"
    )
    out.append("|---|---|---|---|---|---|")
    for _, r in agg_df.iterrows():
        out.append(
            f"| {MODEL_LABELS[r['model']]} | {r[factor_col]} | {int(r['n'])} | "
            f"{fmt(r['median_LD_reactive'])} | {fmt(r['mean_LD_reactive'])} | "
            f"{r['frac_LD_reactive_pos']:.3f} |"
        )
    out.append("")
    return out


lines += factor_table("state_role", by_state_role, "state_role")
lines += factor_table("decl_order", by_decl_order, "decl_order")
lines += factor_table("body_order", by_body_order, "body_order")
lines += factor_table("reactive_body_pos (order-bias 통제 핵심)", by_reactive_body_pos, "reactive_body_pos")

lines.append("## 3. order effect 대 role effect 자동 비교 (초안)")
lines.append("")
lines.append(
    "`range_*`는 해당 factor의 두 레벨 사이 median LD_reactive 차이(max - min)다. "
    "`order_range_exceeds_role_range`가 True면 body_order/reactive_body_pos의 효과가 "
    "state_role 효과보다 크다는 뜻이고, 이는 sync note §9.2의 order-dominated negative 해석에 "
    "더 가깝다는 신호다. **이 표는 자동 비교 초안이며 최종 해석은 연구자가 한다.**"
)
lines.append("")
lines.append(
    "| model | range state_role | range decl_order | range body_order | "
    "range reactive_body_pos | order range > role range | sign consistent (8 combo) |"
)
lines.append("|---|---|---|---|---|---|---|")
for _, r in order_vs_role_flag.iterrows():
    lines.append(
        f"| {MODEL_LABELS[r['model']]} | {fmt(r['range_state_role'])} | "
        f"{fmt(r['range_decl_order'])} | {fmt(r['range_body_order'])} | "
        f"{fmt(r['range_reactive_body_pos'])} | {r['order_range_exceeds_role_range']} | "
        f"{r['median_LD_reactive_sign_consistent_across_8_combos']} |"
    )
lines.append("")

lines.append(
    "## 4. 완전교차 표 (`model x state_role x decl_order x body_order x reactive_body_pos`)"
)
lines.append("")
lines.append(
    "원본은 `reactivity_summary_table.csv`. 모델당 최대 8개 조합(reactive_body_pos는 "
    "state_role과 body_order로 결정되므로 16이 아니라 8)."
)
lines.append("")
lines.append(
    "| model | state_role | decl_order | body_order | reactive_body_pos | n | "
    "median LD_reactive | mean LD_reactive | frac(LD_reactive>0) |"
)
lines.append("|---|---|---|---|---|---|---|---|---|")
for _, r in summary_table.iterrows():
    lines.append(
        f"| {MODEL_LABELS[r['model']]} | {r['state_role']} | {r['decl_order']} | "
        f"{r['body_order']} | {r['reactive_body_pos']} | {int(r['n'])} | "
        f"{fmt(r['median_LD_reactive'])} | {fmt(r['mean_LD_reactive'])} | "
        f"{r['frac_LD_reactive_pos']:.3f} |"
    )
lines.append("")

lines.append("## 5. Auxiliary: close(`]`) 진단 (main 해석에 사용하지 않음)")
lines.append("")
lines.append(
    "| model | n_total | close 계산 성공 | close 계산 실패 | median LD_reactive_vs_close | "
    "median LD_stable_vs_close | frac(reactive < close) | frac(stable < close) |"
)
lines.append("|---|---|---|---|---|---|---|---|")
for _, r in aux_close_summary.iterrows():
    lines.append(
        f"| {MODEL_LABELS[r['model']]} | {int(r['n_total'])} | {int(r['n_close_ok'])} | "
        f"{int(r['n_close_fail'])} | {fmt(r['median_LD_reactive_vs_close'])} | "
        f"{fmt(r['median_LD_stable_vs_close'])} | {fmt(r['frac_reactive_below_close'])} | "
        f"{fmt(r['frac_stable_below_close'])} |"
    )
lines.append("")

lines.append("## 6. 해석 가이드 (sync note §9, 자동 판정 아님)")
lines.append("")
lines.append(
    "- **positive**: §1의 median/frac이 두 모델 모두 양수 방향이고, §2의 세 factor"
    "(state_role / decl_order / body_order) swap에도 부호가 유지되며, §3에서 order range가 "
    "role range를 넘지 않는 경우. 이 경우 허용 claim은 \"useState-form / declared-reactivity "
    "sensitivity가 관찰된다\"까지이며, setterㆍdestructuring confound 한계(§7)를 함께 명시해야 "
    "한다."
)
lines.append(
    "- **order-dominated negative**: §3에서 body_order 또는 reactive_body_pos의 range가 "
    "state_role의 range보다 크면, 모델 예측이 declared-reactivity보다 surface order/copy "
    "salience에 더 가깝다는 신호다."
)
lines.append(
    "- **null/mixed**: §1의 median이 0 근처이거나 모델 간 부호가 불안정하면, 이번 operational "
    "contrast에서 안정적 증거가 없다는 뜻이다."
)
lines.append(
    "- 위 세 갈래 중 어디에 해당하는지, 그리고 최종 메인 claim의 표현은 이 표들을 검토한 "
    "연구자가 결정한다. 이 노트북은 판정하지 않는다."
)
lines.append("")

lines.append("## 7. Limitation")
lines.append("")
lines.append(
    "1. `reactive_var`/`stable_var`는 useState/const 선언 형태에 붙인 라벨이며 reactivity 그 "
    "자체를 측정한 것이 아니다."
)
lines.append(
    "2. positive 결과가 나와도, state_role swap을 하더라도 useState 배정에는 setter(`setX`)와 "
    "destructuring(`const [X, setX] = ...`) 구조가 항상 동반된다. 따라서 positive 결과는 "
    "\"useState 선언 형태에 대한 민감성\"까지만 지지하며, reactivity 자체 때문인지 "
    "setter/destructuring 형태 때문인지는 이 design으로 분리되지 않는다."
)
lines.append(
    "3. `close_target`(`]`) 관련 auxiliary 지표는 tokenization artifact의 영향을 받을 수 있어 "
    "main 결론에 쓰지 않는다."
)
lines.append(
    "4. 이 pilot은 `useEffect` 단일 condition만 다룬다. 다른 syntactic 환경(useLayoutEffect, "
    "alias, non-hook callback-array)으로의 일반화는 baseline diagnostic의 범위이고, 이 "
    "pilot에서는 다루지 않는다."
)
lines.append("5. activation patching은 이 단계에 포함하지 않는다 -- 본 문서는 logit 측정과 집계까지다.")
lines.append("6. 1B 모델(Llama 3.2 1B, Gemma 3 1B-pt) 두 개에 대해서만 측정했다.")
lines.append("")

lines.append("## 8. 출력 파일")
lines.append("")
lines.append(f"- `reactivity_logits_3tokens.csv`: long raw, {len(df_long)} row.")
lines.append(
    f"- `reactivity_wide_per_sample.csv`: pair 단위 wide + factor effect, "
    f"{len(wide_per_sample)} row."
)
lines.append(f"- `reactivity_summary_table.csv`: 완전교차 집계, {len(summary_table)} row.")
lines.append("- `reactivity_summary.md`: 본 문서.")
lines.append("")

with open("reactivity_summary.md", "w") as f:
    f.write("\n".join(lines))

print("wrote reactivity_summary.md")


wrote reactivity_summary.md


## 결론 (작성 시점 -- 미실행)

이 노트북은 작성 시점에 실행되지 않았다(로컬 환경에 GPU·모델 가중치가 없음). 위에서부터 순서대로
셀을 돌리면서 각 단계의 출력, 특히 Step 3의 토큰 검증 통과율과 Step 7–9의 집계 표를 확인하라.

- 메인 결과(§1)와 factor 분해(§2), order-effect 비교(§3)를 함께 봐야 한다 — §1만 보고 positive로
  판단하지 않는다.
- 최종 메인 claim의 표현, 그리고 baseline diagnostic 대비 이 pilot의 위치는 연구자가 결정한다.
- 출력 파일: `reactivity_logits_3tokens.csv`, `reactivity_wide_per_sample.csv`,
  `reactivity_summary_table.csv`, `reactivity_summary.md`.
